In [1]:
# ==========================================
# 1. SETUP & IMPORTS
# ==========================================
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor, GeneralizedLinearRegression
import os

# Initialize Spark
spark = (SparkSession.builder
    .appName("BIXI-Model-Factory")
    .config("spark.sql.session.timeZone", "America/Toronto")
    .getOrCreate())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 23:22:08 WARN Utils: Your hostname, Comanes-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.11 instead (on interface en0)
26/04/17 23:22:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 23:22:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# ==========================================
# 2. CONFIGURATION (Change this for new stations)
# ==========================================
targetStation = "STN_0001"
gold_path = "data/gold/station_flow"
models_save_dir = f"models/{targetStation}/"

# Ensure save directory exists
os.makedirs(models_save_dir, exist_ok=True)



In [ ]:
# ==========================================
# 3. DATA LOADING & TEMPORAL SPLIT
# ==========================================
print(f"Loading data for {targetStation}...")
gold_df = spark.read.parquet(gold_path).filter(F.col("station_id") == targetStation)

# Hardcoded temporal split based on business requirements
split_date = "2025-08-01 00:00:00"

# For the factory, we ONLY need the training data (< 2025-08-01)
train_df = gold_df.filter(F.col("ts_hour") < split_date).cache()
print(f"Training Data Ready: {train_df.count():,} rows. Cutoff date: strictly before {split_date}")


Loading data for STN_0004...


26/04/17 23:22:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training Data Ready: 19,304 rows. Cutoff date: strictly before 2025-08-01 00:00:00


In [4]:
# ==========================================
# 4. FEATURE DEFINITIONS
# ==========================================
EXCLUDE_COLS = ["ts_hour", "station_id", "station_inflow", "station_outflow", "station_netflow"]

# Define numeric and categorical columns
categorical_cols = ["temp_bin"] # Add any other string columns here
numeric_cols = [f.name for f in train_df.schema.fields 
                if f.dataType.typeName() in ['integer', 'long', 'double', 'float'] 
                and f.name not in EXCLUDE_COLS]

# Create standard feature stages
feature_stages = []
for col in categorical_cols:
    feature_stages.append(StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep"))
    feature_stages.append(OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_vec"))

encoded_cols = [f"{col}_vec" for col in categorical_cols]
assembler = VectorAssembler(inputCols=numeric_cols + encoded_cols, outputCol="features", handleInvalid="skip")
feature_stages.append(assembler)

# Add a scaler specifically for the Poisson GLR
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)



In [11]:
# ==========================================
# 5. MODEL ARCHITECTURES & TUNING GRIDS
# ==========================================
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor, GeneralizedLinearRegression
from pyspark.ml.tuning import ParamGridBuilder

# Define the base models 
gbt = GBTRegressor(featuresCol="features", seed=42)
rf = RandomForestRegressor(featuresCol="features", seed=42)
poisson = GeneralizedLinearRegression(featuresCol="scaledFeatures", family="poisson", link="log")

# Build the grids
gbt_grid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [4, 6])          
    .addGrid(gbt.maxIter, [50, 100])        
    .build())

rf_grid = (ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10])      
    .addGrid(rf.numTrees, [50, 100])        
    .build())

poisson_grid = (ParamGridBuilder()
    .addGrid(poisson.regParam, [0.01, 0.1]) 
    .build())

# THE CRITICAL FIX: The nested dictionary
architectures = {
    "GBT": {"model": gbt, "grid": gbt_grid},
    "RF": {"model": rf, "grid": rf_grid},
    "Poisson": {"model": poisson, "grid": poisson_grid}
}

In [ ]:
# ==========================================
# 6. THE BASIC TRAINING LOOP
# ==========================================
for flow_type, target_col in targets.items():
    print(f"\n--- Training {flow_type.upper()} Models ---")
    
    for model_name, algo in architectures.items():
        print(f"Training {model_name}...")
        
        # Set the target column dynamically
        algo.setLabelCol(target_col)
        
        # Build Pipeline (Poisson needs the scaler)
        if model_name == "Poisson":
            pipeline = Pipeline(stages=feature_stages + [scaler, algo])
        else:
            pipeline = Pipeline(stages=feature_stages + [algo])
            
        # Fit the model
        trained_model = pipeline.fit(train_df)
        
        # Save the model to disk (overwrite if exists)
        save_path = os.path.join(models_save_dir, f"{flow_type}_{model_name}")
        trained_model.write().overwrite().save(save_path)
        print(f"✅ Saved to: {save_path}")

print("\n🎉 All models trained and saved successfully!")


--- Training OUTFLOW Models ---
Training GBT...


26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1000.9 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1005.5 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1008.4 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1008.9 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1009.9 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1010.6 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1012.9 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1017.3 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1020.4 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1020.9 KiB
26/04/17 23:30:39 WARN DAGScheduler: Broadcasting large task binary with size 1021.9 KiB
26/04/17 23:30:39 WAR

✅ Saved to: models/STN_0004/outflow_GBT
Training RF...


26/04/17 23:30:47 WARN DAGScheduler: Broadcasting large task binary with size 1431.2 KiB
26/04/17 23:30:48 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/17 23:30:49 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
26/04/17 23:30:50 WARN DAGScheduler: Broadcasting large task binary with size 1139.6 KiB
26/04/17 23:30:51 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB
26/04/17 23:30:52 WARN DAGScheduler: Broadcasting large task binary with size 1770.9 KiB
26/04/17 23:30:53 WARN DAGScheduler: Broadcasting large task binary with size 8.8 MiB
26/04/17 23:30:54 WARN DAGScheduler: Broadcasting large task binary with size 1769.2 KiB
26/04/17 23:30:55 WARN DAGScheduler: Broadcasting large task binary with size 6.4 MiB
26/04/17 23:30:56 WARN DAGScheduler: Broadcasting large task binary with size 1306.6 KiB
26/04/17 23:30:56 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 23:30:58 WARN TaskSetManager: 

✅ Saved to: models/STN_0004/outflow_RF
Training Poisson...
✅ Saved to: models/STN_0004/outflow_Poisson

--- Training INFLOW Models ---
Training GBT...


26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1000.0 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1000.8 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1003.1 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1007.4 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1009.9 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1010.4 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1011.4 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1012.2 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1014.5 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1019.0 KiB
26/04/17 23:31:09 WARN DAGScheduler: Broadcasting large task binary with size 1021.8 KiB
26/04/17 23:31:09 WAR

✅ Saved to: models/STN_0004/inflow_GBT
Training RF...


26/04/17 23:31:18 WARN DAGScheduler: Broadcasting large task binary with size 1431.7 KiB
26/04/17 23:31:19 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/17 23:31:20 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/04/17 23:31:21 WARN DAGScheduler: Broadcasting large task binary with size 1154.7 KiB
26/04/17 23:31:22 WARN DAGScheduler: Broadcasting large task binary with size 7.8 MiB
26/04/17 23:31:23 WARN DAGScheduler: Broadcasting large task binary with size 1770.7 KiB
26/04/17 23:31:24 WARN DAGScheduler: Broadcasting large task binary with size 8.7 MiB
26/04/17 23:31:25 WARN DAGScheduler: Broadcasting large task binary with size 1769.3 KiB
26/04/17 23:31:26 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
26/04/17 23:31:26 WARN DAGScheduler: Broadcasting large task binary with size 1388.1 KiB
26/04/17 23:31:27 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/04/17 23:31:28 WARN TaskSetManager: 

✅ Saved to: models/STN_0004/inflow_RF
Training Poisson...
✅ Saved to: models/STN_0004/inflow_Poisson

🎉 All models trained and saved successfully!


In [12]:
# ==========================================
# 7. TIME-SERIES HYPERPARAMETER TUNING 
# ==========================================

from pyspark.ml.evaluation import RegressionEvaluator

# We split the training data AGAIN to create a strict "Validation" period
# Example: Let's use the month of July 2025 as the tuning validation set
validation_date = "2025-07-01 00:00:00"

# Train the tuning models on everything before July
tune_train_df = train_df.filter(F.col("ts_hour") < validation_date).cache()
# Evaluate the tuning models strictly on July
tune_val_df = train_df.filter(F.col("ts_hour") >= validation_date).cache()

print(f"Tuning Train Size: {tune_train_df.count():,} | Tuning Val Size (July): {tune_val_df.count():,}")

evaluator = RegressionEvaluator(metricName="rmse")

for flow_type, target_col in targets.items():
    print(f"\n--- Tuning {flow_type.upper()} Models ---")
    evaluator.setLabelCol(target_col)
    
    for model_name, components in architectures.items():
        print(f"Starting Time-Series Grid Search for {model_name}...")
        
        algo = components["model"]
        grid = components["grid"]
        algo.setLabelCol(target_col)
        
        best_rmse = float("inf")
        best_params = None
        
        # Loop through every combination of parameters Spark generated
        for param_map in grid:
            # Apply the parameters to the algorithm
            algo_with_params = algo.copy(param_map)
            
            # Build temporary pipeline
            if model_name == "Poisson":
                pipeline = Pipeline(stages=feature_stages + [scaler, algo_with_params])
            else:
                pipeline = Pipeline(stages=feature_stages + [algo_with_params])
            
            # Train on the past (< July)
            temp_model = pipeline.fit(tune_train_df)
            
            # Predict on the future (July)
            predictions = temp_model.transform(tune_val_df)
            current_rmse = evaluator.evaluate(predictions)
            
            # Keep track of the lowest error
            if current_rmse < best_rmse:
                best_rmse = current_rmse
                best_params = param_map
                
        print(f"🏆 Best {model_name} Params found! RMSE: {best_rmse:.3f}")
        
        # FINAL STEP: Retrain the absolute best model on the FULL training set (< August)
        print(f"Retraining final {model_name} on all training data...")
        final_algo = algo.copy(best_params)
        final_pipeline = Pipeline(stages=feature_stages + ([scaler, final_algo] if model_name == "Poisson" else [final_algo]))
        
        best_final_model = final_pipeline.fit(train_df)
        
        # Save to disk
        save_path = os.path.join(models_save_dir, f"{flow_type}_{model_name}_optimized")
        best_final_model.write().overwrite().save(save_path)
        print(f"✅ Saved to: {save_path}")

26/04/17 23:33:32 WARN CacheManager: Asked to cache already cached data.
26/04/17 23:33:32 WARN CacheManager: Asked to cache already cached data.


Tuning Train Size: 18,560 | Tuning Val Size (July): 744

--- Tuning OUTFLOW Models ---
Starting Time-Series Grid Search for GBT...


26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1000.0 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1006.4 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1006.8 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1007.8 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1008.6 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1010.9 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1014.6 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1016.8 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1017.2 KiB
26/04/17 23:34:09 WARN DAGScheduler: Broadcasting large task binary with size 1018.2 KiB
26/04/17 23:34:09 WAR

🏆 Best GBT Params found! RMSE: 6.060
Retraining final GBT on all training data...
✅ Saved to: models/STN_0004/outflow_GBT_optimized
Starting Time-Series Grid Search for RF...


26/04/17 23:34:33 WARN DAGScheduler: Broadcasting large task binary with size 1459.3 KiB
26/04/17 23:34:34 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/17 23:34:35 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/04/17 23:34:36 WARN DAGScheduler: Broadcasting large task binary with size 7.8 MiB
26/04/17 23:34:37 WARN DAGScheduler: Broadcasting large task binary with size 1574.2 KiB
26/04/17 23:34:40 WARN DAGScheduler: Broadcasting large task binary with size 1458.6 KiB
26/04/17 23:34:41 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/17 23:34:43 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/04/17 23:34:44 WARN DAGScheduler: Broadcasting large task binary with size 1130.8 KiB
26/04/17 23:34:45 WARN DAGScheduler: Broadcasting large task binary with size 8.0 MiB
26/04/17 23:34:46 WARN DAGScheduler: Broadcasting large task binary with size 1771.0 KiB
26/04/17 23:34:47 WARN DAGScheduler: Br

...

🏆 Best GBT Params found! RMSE: 4.441
Retraining final GBT on all training data...
✅ Saved to: models/STN_0004/inflow_GBT_optimized
Starting Time-Series Grid Search for RF...


26/04/17 23:36:05 WARN DAGScheduler: Broadcasting large task binary with size 1463.4 KiB
26/04/17 23:36:06 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/17 23:36:07 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/04/17 23:36:07 WARN DAGScheduler: Broadcasting large task binary with size 1019.4 KiB
26/04/17 23:36:08 WARN DAGScheduler: Broadcasting large task binary with size 8.1 MiB
26/04/17 23:36:09 WARN DAGScheduler: Broadcasting large task binary with size 1645.8 KiB
26/04/17 23:36:12 WARN DAGScheduler: Broadcasting large task binary with size 1458.2 KiB
26/04/17 23:36:13 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/17 23:36:14 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/04/17 23:36:15 WARN DAGScheduler: Broadcasting large task binary with size 1158.2 KiB
26/04/17 23:36:16 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB
26/04/17 23:36:17 WARN DAGScheduler: Br

🏆 Best RF Params found! RMSE: 4.418
Retraining final RF on all training data...


26/04/17 23:36:24 WARN DAGScheduler: Broadcasting large task binary with size 1432.4 KiB
26/04/17 23:36:24 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/17 23:36:25 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/04/17 23:36:26 WARN DAGScheduler: Broadcasting large task binary with size 1029.5 KiB
26/04/17 23:36:27 WARN DAGScheduler: Broadcasting large task binary with size 8.1 MiB
26/04/17 23:36:28 WARN DAGScheduler: Broadcasting large task binary with size 1671.1 KiB
26/04/17 23:36:29 WARN TaskSetManager: Stage 13112 contains a task of very large size (1370 KiB). The maximum recommended task size is 1000 KiB.


✅ Saved to: models/STN_0004/inflow_RF_optimized
Starting Time-Series Grid Search for Poisson...
🏆 Best Poisson Params found! RMSE: 8.073
Retraining final Poisson on all training data...
✅ Saved to: models/STN_0004/inflow_Poisson_optimized


In [ ]:
#########
# COMPARISON TABLE OF ALL MODELS
#########

import pandas as pd
import os
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Setup our three mathematical evaluators
rmse_evaluator = RegressionEvaluator(metricName="rmse")
mae_evaluator = RegressionEvaluator(metricName="mae")
r2_evaluator = RegressionEvaluator(metricName="r2")

# Create an empty list to hold our results
evaluation_results = []

print("Evaluating saved models... This might take a moment.")

# 2. Loop through both Inflow and Outflow
for flow_type, target_col in targets.items():
    
    # Tell the evaluators which column represents the "Truth"
    rmse_evaluator.setLabelCol(target_col)
    mae_evaluator.setLabelCol(target_col)
    r2_evaluator.setLabelCol(target_col)
    
    # 3. Loop through the three architectures we trained
    for model_name in ["GBT", "RF", "Poisson"]:
        
        # Construct the path to where the model is saved
        model_path = os.path.join(models_save_dir, f"{flow_type}_{model_name}_optimized")
        
        try:
            # Load the optimized pipeline from disk
            loaded_model = PipelineModel.load(model_path)
            
            # Make predictions on your validation/test set 
            # Note: Replace 'tune_val_df' with whatever dataframe you want to test on!
            predictions = loaded_model.transform(tune_val_df)
            
            # Calculate the metrics
            rmse = rmse_evaluator.evaluate(predictions)
            mae = mae_evaluator.evaluate(predictions)
            r2 = r2_evaluator.evaluate(predictions)
            
            # Store the results
            evaluation_results.append({
                "Flow Type": flow_type.upper(),
                "Model": model_name,
                "RMSE": round(rmse, 3),
                "MAE": round(mae, 3),
                "R-Squared": round(r2, 3)
            })
            
        except Exception as e:
            print(f"Could not evaluate {flow_type} {model_name}. Did it save correctly? Error: {e}")

# 4. Convert the results into a clean Pandas DataFrame for visualization
comparison_df = pd.DataFrame(evaluation_results)

# Sort the table so it's easy to read: Group by Flow Type, then rank by best MAE
comparison_df = comparison_df.sort_values(by=["Flow Type", "MAE"], ascending=[True, True]).reset_index(drop=True)

print("\n🏆 Model Comparison Table 🏆")
display(comparison_df) # Use print(comparison_df) if 'display' doesn't work in your environment

Evaluating saved models... This might take a moment.

🏆 Model Comparison Table 🏆


,Flow Type,Model,RMSE,MAE,R-Squared
0,INFLOW,GBT,3.225,2.430,0.900
1,INFLOW,RF,3.459,2.603,0.885
2,INFLOW,Poisson,7.765,5.318,0.419
3,OUTFLOW,RF,4.368,3.286,0.899
4,OUTFLOW,GBT,4.808,3.585,0.878
5,OUTFLOW,Poisson,10.223,7.020,0.446


# outflow model (RF) optimisation

In [ ]:
#########
# FEATURE IMPORTANCE ANALYSIS (For the best RF model)
#########

import pandas as pd
from pyspark.ml import PipelineModel

import pandas as pd
from pyspark.ml import PipelineModel

# 1. Load the model
model_path = f"models/{targetStation}/outflow_RF_optimized"
print(f"Loading model from: {model_path}...")
loaded_pipeline = PipelineModel.load(model_path)
ml_model = loaded_pipeline.stages[-1]

# 2. Extract the importances array
importances = ml_model.featureImportances.toArray()

# 3. THE FIX: Extract true feature names from PySpark metadata
# We push 1 row of data through the pipeline to generate the metadata
# Note: Using 'tune_val_df' here, but you can use 'train_df' or any df in memory
sample_transformed = loaded_pipeline.transform(tune_val_df.limit(1))

# Get the exact column the model used (usually "features")
features_col_name = ml_model.getFeaturesCol()

# Dig into PySpark's hidden metadata dictionary
feature_attrs = sample_transformed.schema[features_col_name].metadata["ml_attr"]["attrs"]

# Unpack the metadata into a list of (index, name) tuples
extracted_names = []
for attr_group in feature_attrs.values():
    for attr in attr_group:
        extracted_names.append((attr['idx'], attr['name']))

# Sort strictly by index so it matches the importances array perfectly!
extracted_names.sort(key=lambda x: x[0])
true_feature_names = [name for idx, name in extracted_names]

# 4. Build a clean Pandas DataFrame (Now they will be the exact same length!)
importance_df = pd.DataFrame({
    "Feature": true_feature_names,
    "Importance": importances
})

# 5. Calculate the percentage and display
importance_df["Importance_%"] = (importance_df["Importance"] * 100).round(2)
importance_df = importance_df.sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\n🏆 Top 10 Most Important Features:")
display(importance_df.head(10))

print("\n🗑️ The 'Noise' (Bottom 15 Features):")
display(importance_df.tail(15))

In [ ]:
# Extract the top 15 feature names from your ranked dataframe
top_15_features = importance_df["Feature"].head(15).tolist()

print("🎯 Your Top 15 Winning Features:")
print("TOP_15_COLS =", top_15_features)

In [29]:
# ==========================================
# STRICT FEATURE DEFINITIONS (Pruned to Top 15)
# ==========================================

# Paste the list you generated in Step 1 right here:
TOP_15_COLS = [
    "radius200m_outflow_lag1" , 
    "radius200m_inflow_lag1" , 
    "radius100m_inflow_lag1", 
    "station_inflow_lag1" , 
    "hod_cos" , 
    "radius500m_outflow_lag1", 
    "hod" , 
    "radius500m_inflow_lag1", 
    "radius100m_outflow_rollmean6", 
    "radius100m_outflow_lag1", 
    "station_outflow_rollmean6" , 
    "radius100m_inflow_rollmean6" , 
    "radius100m_outflow_rollmean12" , 
    "station_outflow_rollsum6" , 
    "radius200m_outflow_rollmean6" 
]

# Create standard feature stages
feature_stages = []

# NOTE: If any of your categorical columns made the Top 15, 
# you still need to StringIndex and OneHotEncode them here!
categorical_cols = ["temp_bin"] # Keep this if temp_bin is in your Top 15

for col in categorical_cols:
    feature_stages.append(StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep"))
    feature_stages.append(OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_vec"))

# Build the final VectorAssembler strictly from your top 15 list
assembler = VectorAssembler(inputCols=TOP_15_COLS, outputCol="features", handleInvalid="skip")
feature_stages.append(assembler)

# Add the scaler for the Poisson GLR
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

In [30]:
# ==========================================
# 1. SETUP & STRICT TEMPORAL SPLITS
# ==========================================
import os
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

targetStation = "STN_0004" # Change if testing a different station
target_col = "station_outflow"

# Ensure gold_df is loaded beforehand, then filter to the target station
gold_df_stn = gold_df.filter(F.col("station_id") == targetStation)

# Time bounds (Train < Aug 1st, Test >= Aug 1st)
test_date = "2025-08-01 00:00:00"
validation_date = "2025-07-01 00:00:00" # Carving out July for tuning

# Create splits
train_full_df = gold_df_stn.filter(F.col("ts_hour") < test_date).cache()
test_df = gold_df_stn.filter(F.col("ts_hour") >= test_date).cache()

# Splitting train further for tuning
tune_train_df = train_full_df.filter(F.col("ts_hour") < validation_date).cache()
tune_val_df = train_full_df.filter(F.col("ts_hour") >= validation_date).cache()

print(f"Data Ready!")
print(f"Total Training Pool (< Aug): {train_full_df.count():,} rows")
print(f"Testing Data (>= Aug): {test_df.count():,} rows")

# ==========================================
# 2. STRICT TOP 15 FEATURES
# ==========================================
TOP_15_COLS = [
    "radius200m_outflow_lag1", 
    "radius200m_inflow_lag1", 
    "radius100m_inflow_lag1", 
    "station_inflow_lag1", 
    "hod_cos", 
    "radius500m_outflow_lag1", 
    "hod", 
    "radius500m_inflow_lag1", 
    "radius100m_outflow_rollmean6", 
    "radius100m_outflow_lag1", 
    "station_outflow_rollmean6", 
    "radius100m_inflow_rollmean6", 
    "radius100m_outflow_rollmean12", 
    "station_outflow_rollsum6", 
    "radius200m_outflow_rollmean6" 
]

# Because all 15 are numeric, we only need the VectorAssembler!
assembler = VectorAssembler(inputCols=TOP_15_COLS, outputCol="features", handleInvalid="skip")
feature_stages = [assembler]

# ==========================================
# 3. RANDOM FOREST ARCHITECTURE & TUNING
# ==========================================
rf = RandomForestRegressor(featuresCol="features", labelCol=target_col, seed=42)

# The Grid: Test 6 combinations
rf_grid = (ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10, 15])      
    .addGrid(rf.numTrees, [50, 100])        
    .build())

evaluator_rmse = RegressionEvaluator(labelCol=target_col, metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, metricName="mae")

print(f"\n--- Tuning OUTFLOW Random Forest ---")
best_rmse = float("inf")
best_params = None

for param_map in rf_grid:
    rf_with_params = rf.copy(param_map)
    pipeline = Pipeline(stages=feature_stages + [rf_with_params])
    
    # Train on past (< July), evaluate on July
    temp_model = pipeline.fit(tune_train_df)
    predictions = temp_model.transform(tune_val_df)
    current_rmse = evaluator_rmse.evaluate(predictions)
    
    if current_rmse < best_rmse:
        best_rmse = current_rmse
        best_params = param_map

print(f"🏆 Best Tuning RMSE (on July Data): {best_rmse:.3f}")

# ==========================================
# 4. FINAL TRAINING & TEST SET EVALUATION
# ==========================================
print(f"\nRetraining final optimized RF on ALL training data (< August)...")
final_rf = rf.copy(best_params)
final_pipeline = Pipeline(stages=feature_stages + [final_rf])

best_final_model = final_pipeline.fit(train_full_df)

# Evaluate on the REAL Future (>= August 1st)
print(f"Testing model on unseen August+ data...")
test_predictions = best_final_model.transform(test_df)

final_rmse = evaluator_rmse.evaluate(test_predictions)
final_mae = evaluator_mae.evaluate(test_predictions)

print("\n==================================")
print("🚀 FINAL TEST SET RESULTS (OUTFLOW)")
print("==================================")
print(f"Test MAE:  {final_mae:.3f} bikes")
print(f"Test RMSE: {final_rmse:.3f} bikes")
print("==================================")

# (Optional) Save it to disk for the Netflow Evaluator later
save_path = f"models/{targetStation}/outflow_RF_optimized"
best_final_model.write().overwrite().save(save_path)

Data Ready!
Total Training Pool (< Aug): 19,304 rows
Testing Data (>= Aug): 4,417 rows

--- Tuning OUTFLOW Random Forest ---


26/04/18 00:06:23 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:06:23 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:06:23 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:06:23 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:06:26 WARN DAGScheduler: Broadcasting large task binary with size 1144.5 KiB
26/04/18 00:06:27 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/18 00:06:27 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/04/18 00:06:28 WARN DAGScheduler: Broadcasting large task binary with size 6.6 MiB
26/04/18 00:06:28 WARN DAGScheduler: Broadcasting large task binary with size 1564.0 KiB
26/04/18 00:06:30 WARN DAGScheduler: Broadcasting large task binary with size 1140.5 KiB
26/04/18 00:06:31 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/18 00:06:31 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
26/04/18 00

🏆 Best Tuning RMSE (on July Data): 6.158

Retraining final optimized RF on ALL training data (< August)...


26/04/18 00:07:26 WARN DAGScheduler: Broadcasting large task binary with size 1113.3 KiB
26/04/18 00:07:26 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/04/18 00:07:27 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
26/04/18 00:07:28 WARN DAGScheduler: Broadcasting large task binary with size 1123.5 KiB
26/04/18 00:07:28 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/04/18 00:07:29 WARN DAGScheduler: Broadcasting large task binary with size 1953.3 KiB
26/04/18 00:07:30 WARN DAGScheduler: Broadcasting large task binary with size 12.9 MiB
26/04/18 00:07:31 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


Testing model on unseen August+ data...

🚀 FINAL TEST SET RESULTS (OUTFLOW)
Test MAE:  3.148 bikes
Test RMSE: 4.954 bikes


26/04/18 00:07:32 WARN TaskSetManager: Stage 13765 contains a task of very large size (2602 KiB). The maximum recommended task size is 1000 KiB.


In [31]:
### adding feature interactions to try improving model ###

from featureinteractions import add_misery_index, add_temporal_context

from featureinteractions import add_misery_index, add_temporal_context
import pyspark.sql.functions as F

# 1. Apply interactions with a CAST for the boolean column
gold_df_stn = (gold_df.filter(F.col("station_id") == targetStation)
    # Misery index works fine (likely two numbers)
    .transform(lambda df: add_misery_index(df, temp_col="temp", precip_col="precip"))
    
    # THE FIX: Cast is_weekday to "int" so Spark can multiply it by the hour
    .withColumn("is_weekday", F.col("is_weekday").cast("int"))
    
    # Now the function will work perfectly!
    .transform(lambda df: add_temporal_context(df, hour_col="hod", weekend_col="is_weekday")) 
)

# 2. Add these to your potential winners
POTENTIAL_COLS = TOP_15_COLS + ["temp_precip_int", "hod_is_weekend_int"]

print("Interactions created successfully!")

# Now proceed with your splits as before
train_full_df = gold_df_stn.filter(F.col("ts_hour") < test_date).cache()
tune_train_df = train_full_df.filter(F.col("ts_hour") < validation_date).cache()
tune_val_df = train_full_df.filter(F.col("ts_hour") >= validation_date).cache()
print("Training and validation sets created successfully!")

Interactions created successfully!
Training and validation sets created successfully!


26/04/18 00:07:33 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:07:33 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:07:33 WARN CacheManager: Asked to cache already cached data.


In [32]:
import pandas as pd
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator

# 1. SETUP - Use the exact same path where you just saved the optimized RF
targetStation = "STN_0004"
test_path = f"models/{targetStation}/outflow_RF_optimized"

# Ensure your test_df is ready (the one >= 2025-08-01)
print(f"Loading fresh model for {targetStation}...")
final_rf_model = PipelineModel.load(test_path)

# 2. GENERATE PREDICTIONS
# This uses the 15 features (including interactions) to predict the future
test_predictions = final_rf_model.transform(test_df)

# 3. CALCULATE METRICS
evaluator_rmse = RegressionEvaluator(labelCol="station_outflow", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="station_outflow", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="station_outflow", metricName="r2")

rmse = evaluator_rmse.evaluate(test_predictions)
mae = evaluator_mae.evaluate(test_predictions)
r2 = evaluator_r2.evaluate(test_predictions)

# 4. DISPLAY THE WINNING STATS
results_data = {
    "Metric": ["MAE (Mean Absolute Error)", "RMSE (Root Mean Squared Error)", "R-Squared"],
    "Value": [round(mae, 4), round(rmse, 4), round(r2, 4)]
}

results_df = pd.DataFrame(results_data)

print("\n🚀 NEWLY TRAINED MODEL RESULTS (OUTFLOW)")
print("==========================================")
display(results_df)

if mae < 2.5:
    print(f"\n✅ SUCCESS! Your MAE is {mae:.3f}. You smashed the 2.xx goal!")
else:
    print(f"\n📈 Current MAE: {mae:.3f}. We are getting closer!")

Loading fresh model for STN_0004...

🚀 NEWLY TRAINED MODEL RESULTS (OUTFLOW)


,Metric,Value
0,MAE (Mean Absolute Error),3.1476
1,RMSE (Root Mean Squared Error),4.9542
2,R-Squared,0.8576



📈 Current MAE: 3.148. We are getting closer!


feature interactions did not better the MAE.

# Optimising best inflow model (GBT)

In [ ]:
#########
# FEATURE IMPORTANCE ANALYSIS (For the best GBT model)
#########

import pandas as pd
from pyspark.ml import PipelineModel

import pandas as pd
from pyspark.ml import PipelineModel

# 1. Load the model
model_path = f"models/{targetStation}/inflow_GBT_optimized"
print(f"Loading model from: {model_path}...")
loaded_pipeline = PipelineModel.load(model_path)
ml_model = loaded_pipeline.stages[-1]

# 2. Extract the importances array
importances = ml_model.featureImportances.toArray()

# 3. THE FIX: Extract true feature names from PySpark metadata
# We push 1 row of data through the pipeline to generate the metadata
# Note: Using 'tune_val_df' here, but you can use 'train_df' or any df in memory
sample_transformed = loaded_pipeline.transform(tune_val_df.limit(1))

# Get the exact column the model used (usually "features")
features_col_name = ml_model.getFeaturesCol()

# Dig into PySpark's hidden metadata dictionary
feature_attrs = sample_transformed.schema[features_col_name].metadata["ml_attr"]["attrs"]

# Unpack the metadata into a list of (index, name) tuples
extracted_names = []
for attr_group in feature_attrs.values():
    for attr in attr_group:
        extracted_names.append((attr['idx'], attr['name']))

# Sort strictly by index so it matches the importances array perfectly!
extracted_names.sort(key=lambda x: x[0])
true_feature_names = [name for idx, name in extracted_names]

# 4. Build a clean Pandas DataFrame (Now they will be the exact same length!)
importance_df = pd.DataFrame({
    "Feature": true_feature_names,
    "Importance": importances
})

# 5. Calculate the percentage and display
importance_df["Importance_%"] = (importance_df["Importance"] * 100).round(2)
importance_df = importance_df.sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\n🏆 Top 10 Most Important Features:")
display(importance_df.head(10))

print("\n🗑️ The 'Noise' (Bottom 15 Features):")
display(importance_df.tail(15))

Loading model from: models/STN_0004/inflow_GBT_optimized...

🏆 Top 10 Most Important Features:


,Feature,Importance,Importance_%
0,radius200m_inflow_lag1,0.328640,32.86
1,radius100m_inflow_rollmean6,0.089254,8.93
2,hod,0.087207,8.72
3,hod_cos,0.049770,4.98
4,radius500m_inflow_lag1,0.042828,4.28
5,radius100m_inflow_rollmean12,0.039454,3.95
6,dow_cos,0.029061,2.91
7,radius100m_inflow_lag1,0.025385,2.54
8,radius100m_outflow_lag1,0.024366,2.44
9,radius500m_outflow_lag1,0.023761,2.38



🗑️ The 'Noise' (Bottom 15 Features):


,Feature,Importance,Importance_%
52,radius100m_outflow_rollsum12,0.0,0.0
53,radius200m_inflow_rollsum6,0.0,0.0
54,radius200m_outflow_rollsum6,0.0,0.0
55,radius200m_inflow_rollsum12,0.0,0.0
56,radius200m_outflow_rollsum12,0.0,0.0
57,radius500m_inflow_rollsum6,0.0,0.0
58,radius500m_inflow_rollsum12,0.0,0.0
59,radius500m_outflow_rollsum12,0.0,0.0
60,station_inflow_lag1,0.0,0.0
61,station_outflow_lag1,0.0,0.0


In [34]:
# Extract the top 15 feature names from your ranked dataframe
top_15_features = importance_df["Feature"].head(15).tolist()

print("🎯 Your Top 15 Winning Features:")
print("TOP_15_COLS =", top_15_features)

🎯 Your Top 15 Winning Features:
TOP_15_COLS = ['radius200m_inflow_lag1', 'radius100m_inflow_rollmean6', 'hod', 'hod_cos', 'radius500m_inflow_lag1', 'radius100m_inflow_rollmean12', 'dow_cos', 'radius100m_inflow_lag1', 'radius100m_outflow_lag1', 'radius500m_outflow_lag1', 'radius200m_outflow_lag1', 'temp', 'radius500m_outflow_lag12', 'dow', 'radius500m_outflow_rollmean6']


In [35]:
# ==========================================
# 1. SETUP & TARGET CHANGE
# ==========================================
from pyspark.ml.regression import GBTRegressor # Need to import GBT

targetStation = "STN_0004" 
target_col = "station_inflow" # Changed to Inflow

# Ensure gold_df is loaded beforehand, then filter to the target station
gold_df_stn = gold_df.filter(F.col("station_id") == targetStation)

# Time bounds (Train < Aug 1st, Test >= Aug 1st)
test_date = "2025-08-01 00:00:00"
validation_date = "2025-07-01 00:00:00" # Carving out July for tuning

# Create splits
train_full_df = gold_df_stn.filter(F.col("ts_hour") < test_date).cache()
test_df = gold_df_stn.filter(F.col("ts_hour") >= test_date).cache()

# Splitting train further for tuning
tune_train_df = train_full_df.filter(F.col("ts_hour") < validation_date).cache()
tune_val_df = train_full_df.filter(F.col("ts_hour") >= validation_date).cache()

print(f"Data Ready!")
print(f"Total Training Pool (< Aug): {train_full_df.count():,} rows")
print(f"Testing Data (>= Aug): {test_df.count():,} rows")

# ==========================================
# 2. FEATURE SELECTION (INFLOW)
# ==========================================
TOP_15_INFLOW_COLS = [
    "radius200m_inflow_lag1", 
    "radius100m_inflow_rollmean6", 
    "hod", 
    "hod_cos", 
    "radius500m_inflow_lag1", 
    "radius100m_inflow_rollmean12", 
    "dow_cos", 
    "radius100m_inflow_lag1", 
    "radius100m_outflow_lag1",
    "radius500m_outflow_lag1", 
    "radius200m_outflow_lag1",
    "temp", 
    "radius500m_outflow_lag12", 
    "dow", 
    "radius500m_outflow_rollmean6"
]

assembler = VectorAssembler(inputCols=TOP_15_INFLOW_COLS, outputCol="features", handleInvalid="skip")
feature_stages = [assembler]

# ==========================================
# 3. GBT ARCHITECTURE & TUNING
# ==========================================
gbt = GBTRegressor(featuresCol="features", labelCol=target_col, seed=42)

# GBTs tune differently than RFs
gbt_grid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [4, 6])          # Depth of trees
    .addGrid(gbt.maxIter, [50, 100])        # Number of boosting rounds
    .addGrid(gbt.stepSize, [0.05, 0.1])     # Learning rate
    .build())

evaluator_rmse = RegressionEvaluator(labelCol=target_col, metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, metricName="mae")

print(f"\n--- Tuning INFLOW GBT ---")
best_rmse = float("inf")
best_params = None

for param_map in gbt_grid:
    gbt_with_params = gbt.copy(param_map)
    pipeline = Pipeline(stages=feature_stages + [gbt_with_params])
    
    # Train/Val logic remains the same
    temp_model = pipeline.fit(tune_train_df)
    predictions = temp_model.transform(tune_val_df)
    current_rmse = evaluator_rmse.evaluate(predictions)
    
    if current_rmse < best_rmse:
        best_rmse = current_rmse
        best_params = param_map

# ==========================================
# 4. FINAL TRAINING & SAVING
# ==========================================
final_gbt = gbt.copy(best_params)
final_pipeline = Pipeline(stages=feature_stages + [final_gbt])
best_final_inflow_model = final_pipeline.fit(train_full_df)

# Evaluate on August+ Data
test_predictions = best_final_inflow_model.transform(test_df)
print(f"Test MAE: {evaluator_mae.evaluate(test_predictions):.3f}")

# Save as inflow_GBT_optimized
save_path = f"models/{targetStation}/inflow_GBT_optimized"
best_final_inflow_model.write().overwrite().save(save_path)

26/04/18 00:13:38 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:13:38 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:13:38 WARN CacheManager: Asked to cache already cached data.
26/04/18 00:13:38 WARN CacheManager: Asked to cache already cached data.


Data Ready!
Total Training Pool (< Aug): 19,304 rows
Testing Data (>= Aug): 4,417 rows

--- Tuning INFLOW GBT ---


26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1000.8 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1004.4 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1005.4 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1006.2 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1008.5 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1012.4 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1014.7 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1015.1 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1016.1 KiB
26/04/18 00:14:21 WARN DAGScheduler: Broadcasting large task binary with size 1016.9 KiB
26/04/18 00:14:21 WAR

Test MAE: 2.328
